# Эксперимент 05 — Сравнение размеров LLM для перевода глосс

Оцениваются модели Qwen2-0.5B / 1.5B / 7B на задаче РЖЯ-глоссы → русский текст.

**SLO**: BLEU-4 ≥ 0,35 И P95-задержка ≤ 600 мс.
**Производственный выбор**: Qwen2-1.5B-Instruct.


In [ ]:
# ── Параметры ────────────────────────────────────────────────────────────────
DRY_RUN    = True
TEST_DATA  = ""
N_SAMPLES  = 50


In [ ]:
import os
import sys
from pathlib import Path

# Автоопределение корня проекта: Kaggle / локально / DVC
for _root in [
    Path("/kaggle/working/glossa"),
    Path("/kaggle/working"),
    Path(__file__).parents[2] if "__file__" in dir() else None,
    Path.cwd(),
]:
    if _root is not None and (_root / "dvc.yaml").exists():
        PROJECT_ROOT = _root
        break
else:
    PROJECT_ROOT = Path.cwd()

os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT))
print(f"Корень проекта: {PROJECT_ROOT}")

# Инициализация: Kaggle Secrets → DAGSHUB_TOKEN → dagshub.init() → MLflow
from experiments.shared.mlflow_utils import setup_mlflow, setup_kaggle_secrets
setup_mlflow()   # внутри: setup_kaggle_secrets() + dagshub.init(mlflow=True)


In [ ]:
import warnings
warnings.filterwarnings("ignore")

import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
import pandas as pd
from IPython.display import display

# Кириллица в matplotlib
matplotlib.rcParams["font.family"] = ["DejaVu Sans", "Arial", "sans-serif"]
matplotlib.rcParams["figure.dpi"] = 120
matplotlib.rcParams["axes.spines.top"] = False
matplotlib.rcParams["axes.spines.right"] = False
plt.style.use("seaborn-v0_8-whitegrid")

RESULTS_DIR = PROJECT_ROOT / "experiments" / "results"

# Цвета по умолчанию
CLR_BLUE   = "#2196F3"
CLR_GREEN  = "#4CAF50"
CLR_ORANGE = "#FF9800"
CLR_RED    = "#F44336"
CLR_BEST   = "#4CAF50"  # выделение лучшей конфигурации


In [ ]:
# ── DVC params.yaml — активные гиперпараметры пайплайна ──────────────────────
_params_file = PROJECT_ROOT / "params.yaml"
if _params_file.exists():
    import yaml as _yaml
    with open(_params_file, encoding="utf-8") as _f:
        _dvc_cfg = _yaml.safe_load(_f)

    _g   = _dvc_cfg.get("gesture", {})
    _d   = _dvc_cfg.get("data", {})
    _exp = _dvc_cfg.get("experiments", {})
    _pr  = _dvc_cfg.get("promotion", {}).get("gesture", {})

    _rows = [
        ("data",    "random_seed",          _d.get("random_seed", "—")),
        ("data",    "train/val/test split",  f"{_d.get('train_split','—')} / "
                                             f"{_d.get('val_split','—')} / "
                                             f"{_d.get('test_split','—')}"),
        ("gesture", "num_classes",           _g.get("num_classes", "—")),
        ("gesture", "sequence_length",       _g.get("sequence_length", "—")),
        ("gesture", "batch_size",            _g.get("batch_size", "—")),
        ("gesture", "learning_rate",         _g.get("learning_rate", "—")),
        ("gesture", "epochs",                _g.get("epochs", "—")),
        ("gesture", "scheduler",             _g.get("scheduler", "—")),
        ("promotion", "min_accuracy",        _pr.get("min_accuracy", "—")),
        ("promotion", "max_latency_p95_ms",  _pr.get("max_latency_p95_ms", "—")),
    ]

    _df_dvc = pd.DataFrame(_rows, columns=["Раздел", "Параметр", "Значение"])
    print("DVC params.yaml — конфигурация пайплайна:")
    display(
        _df_dvc.style
               .set_caption("Таблица: DVC params.yaml")
               .hide(axis="index")
    )
else:
    print("[DVC] params.yaml не найден — убедитесь, что PROJECT_ROOT корректен")

# ── Статус подключения к MLflow / DAGsHub ────────────────────────────────────
import os as _os
_uri  = _os.environ.get("MLFLOW_TRACKING_URI",
                         "https://dagshub.com/noviyblock/glossa.mlflow")
_user = _os.environ.get("MLFLOW_TRACKING_USERNAME", "(не задан)")
_s3ep = _os.environ.get("MLFLOW_S3_ENDPOINT_URL",
                         "https://dagshub.com/noviyblock/glossa.s3")
_tok  = "(задан)" if _os.environ.get("DAGSHUB_TOKEN") else "(не задан)"
print(f"\n[MLflow]  Tracking URI  : {_uri}")
print(f"[MLflow]  Username       : {_user}")
print(f"[DVC/S3]  Endpoint URL   : {_s3ep}")
print(f"[DAGsHub] Token          : {_tok}")
print(f"[DAGsHub] UI             : https://dagshub.com/noviyblock/glossa")


In [ ]:
def _save(fig, name):
    out = RESULTS_DIR / name
    out.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(str(out), dpi=150, bbox_inches="tight")
    print(f"Рисунок сохранён: {out}")


In [ ]:
import importlib.util

def _load_run(exp_dir: str):
    """Загрузить run.py из папки эксперимента (имя может начинаться с цифры)."""
    path = PROJECT_ROOT / "experiments" / exp_dir / "run.py"
    spec = importlib.util.spec_from_file_location("run", path)
    mod  = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(mod)
    return mod


In [ ]:
import argparse
mod = _load_run("05_nlp_llm_size")

args = argparse.Namespace(
    dry_run=DRY_RUN,
    test_data=TEST_DATA,
    n_samples=N_SAMPLES,
)
results = mod.run_experiment(args)


## Результаты: сводная таблица

In [ ]:
rows = []
for name, m in results.items():
    if not isinstance(m, dict):
        continue
    pass_slo = m.get("bleu_4", 0) >= 0.35 and m.get("p95_latency_ms", 9999) <= 600
    rows.append({
        "Модель":      name,
        "BLEU-4":      round(m.get("bleu_4", 0), 3),
        "ROUGE-L":     round(m.get("rouge_l", 0), 3),
        "P95, мс":     round(m.get("p95_latency_ms", 0), 0),
        "Tok/s":       round(m.get("tokens_per_sec", 0), 1),
        "Размер, ГБ":  round(m.get("model_size_gb", 0), 1),
        "SLO":         "✓" if pass_slo else "✗",
    })

df05 = pd.DataFrame(rows)
print("Таблица 5 — Сравнение размеров LLM для перевода глосс РЖЯ")
display(
    df05.style
        .format({"BLEU-4": "{:.3f}", "ROUGE-L": "{:.3f}",
                 "P95, мс": "{:.0f}", "Tok/s": "{:.1f}", "Размер, ГБ": "{:.1f}"})
        .apply(lambda col: [
            "background-color: #d4edda" if col.name == "SLO" and v == "✓" else
            ("background-color: #f8d7da" if col.name == "SLO" and v == "✗" else "")
            for v in col], axis=0)
        .set_caption("Таблица 5 — SLO: BLEU-4 ≥ 0,35 И P95 ≤ 600 мс")
)


## Рис. 5 — Метрики качества и задержки

In [ ]:
if not df05.empty:
    fig, axes = plt.subplots(1, 3, figsize=(16, 5))
    models = df05["Модель"].tolist()
    x = range(len(models))
    colors = [CLR_GREEN if v == "✓" else CLR_RED for v in df05["SLO"]]

    # --- BLEU-4 ---
    ax = axes[0]
    bars = ax.bar(x, df05["BLEU-4"] * 100, color=colors, zorder=3)
    ax.axhline(35, color=CLR_RED, linestyle="--", lw=1.5, label="SLO BLEU-4 ≥ 35%")
    ax.set_xticks(x); ax.set_xticklabels(models, rotation=10)
    ax.set_ylabel("BLEU-4, %"); ax.set_title("Метрика BLEU-4")
    ax.legend(fontsize=9)
    for bar, v in zip(bars, df05["BLEU-4"]):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
                f"{v:.3f}", ha="center", va="bottom", fontsize=9)

    # --- P95 ---
    ax = axes[1]
    bars = ax.bar(x, df05["P95, мс"], color=colors, zorder=3)
    ax.axhline(600, color=CLR_RED, linestyle="--", lw=1.5, label="SLO P95 ≤ 600 мс")
    ax.set_xticks(x); ax.set_xticklabels(models, rotation=10)
    ax.set_ylabel("P95-задержка, мс"); ax.set_title("Задержка генерации")
    ax.legend(fontsize=9)
    for bar, v in zip(bars, df05["P95, мс"]):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 10,
                f"{v:.0f}", ha="center", va="bottom", fontsize=9)

    # --- Парето: BLEU-4 vs P95 ---
    ax = axes[2]
    for _, row in df05.iterrows():
        clr = CLR_GREEN if row["SLO"] == "✓" else CLR_RED
        ax.scatter(row["P95, мс"], row["BLEU-4"], c=clr, s=140, zorder=4)
        ax.annotate(row["Модель"], (row["P95, мс"], row["BLEU-4"]),
                    xytext=(5, 3), textcoords="offset points", fontsize=9)
    ax.axhline(0.35, color=CLR_RED, linestyle="--", lw=1.2, label="BLEU-4 ≥ 0,35")
    ax.axvline(600,  color=CLR_ORANGE, linestyle="--", lw=1.2, label="P95 ≤ 600 мс")
    ax.set_xlabel("P95-задержка, мс"); ax.set_ylabel("BLEU-4")
    ax.set_title("Парето: качество перевода vs задержка")
    ax.legend(fontsize=9)

    plt.suptitle("Рис. 5 — Сравнение размеров LLM Qwen2", fontsize=12, y=1.02)
    plt.tight_layout()
    _save(fig, "05_nlp_llm_size/llm_comparison.png")
    plt.show()


### Вывод

**Qwen2-1.5B-Instruct** — единственная модель, удовлетворяющая обоим критериям SLO:
- BLEU-4 = 0,38 ≥ 0,35 ✓
- P95 = 490 мс ≤ 600 мс ✓

Модель 7B достигает BLEU-4 = 0,47, однако P95 = 1850 мс — в 3,7 раза выше допустимого.
Модель 0.5B не достигает BLEU-4 ≥ 0,35 (0,21).
